# វគ្គសិក្សា 02 - ការស្វែងយល់អំពី Microsoft Agent Framework

**Microsoft Agent Framework (MAF)** គឺជា​រចនាសម្ព័ន្ធមួយសម្រាប់សាងសង់ភ្នាក់ងារ AI។ វាផ្តល់នូវរចនាសម្ព័ន្ធស្អាត និងអាចចែកជាធាតុខុសៗគ្នា ដែលមានធាតុគ្រឹះចំនួនបួន៖

- **Client** – ភ្ជាប់ទៅកាន់ចំណុចបញ្ចប់របស់ម៉ូឌែល AI ហើយគ្រប់គ្រងការទំនាក់ទំនង
- **Agent** – ដាក់បន្ទូល client ជាមួយនឹងការណែនាំ និងការកំណត់ឧបករណ៍
- **Tools** – ពង្រីកសមត្ថភាពរបស់ agent ជាមួយមុខងារផ្ទាល់ខ្លួនដែលម៉ូឌែលអាចហៅបាន
- **Session** – រក្សាប្រវត្តិការសន្ទនាសម្រាប់អន្តរកម្មច្រើនជុំ

នៅក្នុងវគ្គនេះ យើងនឹងសាងសង់ **ភ្នាក់ងារកក់ដំណើរ** ដែលពិនិត្យភាពទំនេរនៃគោលដៅដោយប្រើគំនិតទាំងនេះ។


## ការដំឡើង


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ការយល់ដឹងអំពីស្ថាបត្យកម្មរចនាសម្ព័ន្ធ Agent Framework

Microsoft Agent Framework អនុវត្ត​ស្ថាបត្យកម្មដែលមាន​ស្រទាប់៖

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Client** – `AzureAIProjectAgentProvider` មួយភ្ជាប់ទៅកាន់ការបង្ហោះ Azure OpenAI។ វាទទួលខុសត្រូវសម្រាប់ការផ្ទៀងផ្ទាត់អត្តសញ្ញាណ, ទ្រង់ទ្រាយសំណើ, និងការវិភាគការឆ្លើយតប។
2. **Agent** – ត្រូវបានបង្កើតពី client តាមរយៈ `provider.create_agent()`, agent នោះផ្សំការចូលប្រើម៉ូឌែលជាមួយនឹងសេចក្តីណែនាំ (system prompt) និងឧបករណ៍។
3. **Tools** – មុខងារ Python ដែលពាក់ស្លាក `@tool` ដែល agent អាចហៅដើម្បីអនុវត្តសកម្មភាព ឬយកទិន្នន័យ។
4. **Session** – វត្ថុ `AgentSession` (បង្កើតតាម `agent.create_session()`) ដែលរក្សាប្រវត្តិការពិភាក្សា អនុញ្ញាតឱ្យមានការពីភាក្សាជាច្រើនជុំ ដែល agent ចាំបរិបទមុនៗ។

ចាប់ផ្តើមសាងសង់ស្រទាប់នីមួយៗជំហានៗ។


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ការបន្ថែមឧបករណ៍ដោយប្រើ @tool decorator

ឧបករណ៍អនុញ្ញាតឱ្យភ្នាក់ងារធ្វើសកម្មភាពដែលលើសពីការបង្កើតអត្ថបទ។ ដេកូរេត័រ `@tool` បម្លែងអនុគមន៍ Python ទូទៅមួយឲ្យទៅជា​អ្វីមួយដែលភ្នាក់ងារអាចហៅបាន។

ចំណុចសំខាន់ៗ:
- ប្រើ `Annotated[type, "description"]` ដូច្នេះម៉ូឌែលនឹងយល់ពីប៉ារ៉ាមែត្រនីមួយៗ។
- docstring ក្លាយទៅជា​ពិពណ៌នាឧបករណ៍ដែលម៉ូឌែលឃើញ។
- `approval_mode="never_require"` មានន័យថាឧបករណ៍នឹងរត់ដោយស្វ័យប្រវត្តិដោយគ្មានការបញ្ជាក់ពីអ្នកប្រើ។


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## បង្កើតភ្នាក់ងារជាមួយឧបករណ៍

ឥឡូវនេះយើងបញ្ចូល client, instructions, និង tools ទៅក្នុងភ្នាក់ងារ។ ពាក្យ `instructions` មានតួនាទីជាសេចក្តីណែនាំរបស់ប្រព័ន្ធ — វាកំណត់អត្តសញ្ញាណ និងឥរិយាបថរបស់ភ្នាក់ងារ។


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ការសន្ទនាច្រើនជុំជាមួយសម័យ

`AgentSession` (បានបង្កើតតាមរយៈ `agent.create_session()`) ធ្វើការតាមដានសារទាំងអស់ក្នុងការសន្ទនា។ ដោយផ្ញើសម័យដូចគ្នាទៅនឹងគ្រប់ការហៅ `agent.run()` អេជិនអាចចូលដំណើរការប្រវត្តិសន្ទនាពេញលេញ ហើយអាចយោងទៅកាន់សារមុនៗបាន។

យើងផ្ញើ `tools=[check_destination_availability]` ដូច្នេះ អេជិនអាចហៅឧបករណ៍ពិនិត្យភាពអាចប្រើបានរបស់យើងក្នុងរាល់ជុំ។


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## សង្ខេប

ក្នុងមេរៀននេះ អ្នកបានស្វែងយល់ពីមូលដ្ឋានសំខាន់ចំនួនបួននៃ Microsoft Agent Framework:

| Concept | What You Learned |
|---------|------------------|
| **Client** | `AzureAIProjectAgentProvider` ភ្ជាប់ទៅ Azure OpenAI ដោយការផ្ទៀងផ្ទាត់ដោយសក្ខីបត្រ |
| **Agent** | `provider.create_agent()` ចងកញ្ចប់ការតភ្ជាប់ម៉ូឌែល ជាមួយសេចក្តីណែនាំ និងឈ្មោះ |
| **Tools** | តុបតែង `@tool` បង្ហាញមុខងារ Python ដែលភ្នាក់ងារអាចហៅបាន |
| **Session** | `agent.create_session()` រក្សាប្រវត្តិសន្ទនាក្នុងចន្លោះជំនួបច្រើន |

ធាតុសាងសង់ទាំងនេះរួមគ្នា ដើម្បីបង្កើតភ្នាក់ងារ ដែលអាចរាប់សន្ទនាបែបធម្មជាតិ, ហៅមុខងារផ្ទៃខាងក្រៅបាន, និងរក្សាបរិបទ — មូលដ្ឋានសម្រាប់លំនាំភ្នាក់ងារដែលកាន់តែទន់ភ្លន់ក្នុងមេរៀនបន្ទាប់។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការមិនទទួលខុសត្រូវ**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាកម្មបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ទោះយ៉ាងណា ខណៈពេលដែលយើងខិតខំក្នុងការធានាឱ្យមានភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថា ការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាដើមគួរត្រូវបានចាត់ទុកថាជាប្រភពផ្លូវការ។ សម្រាប់ព័ត៌មានសំខាន់ៗ យើងអនុញ្ញាតអោយមានការបកប្រែដោយមនុស្សជំនាញវិជ្ជាជីវៈ។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
